In [1]:
bucket_name ='ibk-discovery-comercial-us-east-1-654654352211-data'
model_prefix = 'discovery/comercial/sanherna/PLAFT/PJ/MINORISTA'

In [2]:
import boto3
import sagemaker
import sagemaker
from sagemaker.sklearn.processing import SKLearnProcessor
from sagemaker.processing import ProcessingInput, ProcessingOutput

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/__init__.py:86: SageMakerV2DeprecationWarning: You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation()
You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


In [3]:
import os
import boto3
from sagemaker import get_execution_role

account = boto3.client('sts').get_caller_identity()['Account']
image_uri = f'{account}.dkr.ecr.us-east-1.amazonaws.com/sagemaker-python3:3.8.15-cpu',
#'763104351884.dkr.ecr.us-east-1.amazonaws.com/pytorch-inference:2.3.0-cpu-py311-ubuntu20.04-sagemaker'
#'763104351884.dkr.ecr.us-east-1.amazonaws.com/pytorch-inference:2.3.0-cpu-py311-ubuntu20.04-sagemaker' #f'{account}.dkr.ecr.us-east-1.amazonaws.com/sagemaker-python3:3.8.15-cpu'
role_arn = get_execution_role()

os.environ['I_TEAM_RETAIL'], os.environ['I_CC_RETAIL'] = 'DS RETAIL', '9946100000'
os.environ['I_TEAM_RIESGOS'], os.environ['I_CC_RIESGOS'] = 'DS RIESGOS', '9810200000'

team = 'RETAIL' # TODO 1: Colocar nombre del equipo: RETAIL, RIESGOS
name_ds = 'Hernandez Santiago' # TODO 2: Colocar mis apellidos y nombres
account = boto3.client('sts').get_caller_identity()['Account']

In [4]:
tags = [
    {'Key': 'I_RESPONSABLE_LT', 'Value': name_ds},
    {'Key': 'I_APLICACION', 'Value': 'SDLF'},
    {'Key': 'I_PROYECTO', 'Value': 'SDLF'},
    {'Key': 'I_AMBIENTE', 'Value': 'DEV'},
    {'Key': 'I_CUENTA', 'Value': account},
    {'Key': 'I_SIGLA', 'Value': 'SAN'},
    {'Key': 'I_TEAM', 'Value': os.environ[f'I_TEAM_{team}']},
    {'Key': 'I_CC', 'Value': os.environ[f'I_CC_{team}']},
]

In [5]:
role = sagemaker.get_execution_role()

In [6]:
sklearn_processor = SKLearnProcessor(framework_version='0.20.0',
                                     base_job_name= 'Digitalizacion',
                                     instance_type='ml.r5.12xlarge',
                                     role=role,
                                     tags=tags,
                                     instance_count=1,volume_size_in_gb=30)

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/processing.py:138: SageMakerV2DeprecationWarning: SKLearnProcessor is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `DataProcessor` (`from sagemaker.mlops.processing import DataProcessor`).
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation(
SKLearnProcessor is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `DataProcessor` (`from sagemaker.mlops.processing import DataProcessor`).
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


In [7]:
import boto3
import sagemaker
from sagemaker import image_uris
from sagemaker.inputs import TrainingInput
from sagemaker.estimator import Estimator
from sagemaker.tuner import HyperparameterTuner, ContinuousParameter, IntegerParameter

# Parámetros base
bucket_name = 'ibk-discovery-comercial-us-east-1-654654352211-data'
model_prefix = 'discovery/comercial/sanherna/PLAFT/PJ/MINORISTA'

# Obtener imagen de XGBoost compatible con SHAP
xgb_image = image_uris.retrieve(framework='xgboost', region=boto3.Session().region_name, version='1.3-1')

# Crear el estimador
xgb = Estimator(
    image_uri=xgb_image,
    role=sagemaker.get_execution_role(),
    instance_count=1,
    instance_type='ml.m5.4xlarge',
    output_path=f's3://{bucket_name}/{model_prefix}/MODEL/output',
    sagemaker_session=sagemaker.Session()
)

# Hiperparámetros base
xgb.set_hyperparameters(
    eval_metric='auc',
    objective='binary:logistic',
    scale_pos_weight=494,
    early_stopping_rounds=200,
    num_round=1000
)

# Espacio de búsqueda
hyperparameter_ranges = {
    'max_depth': IntegerParameter(3, 9),                # igual
    'eta': ContinuousParameter(0.01, 0.25),             # learning_rate
    'subsample': ContinuousParameter(0.6, 1.0),         # igual
    'colsample_bytree': ContinuousParameter(0.6, 1.0),  # igual
    'gamma': ContinuousParameter(0, 7),                 # igual
    'min_child_weight': IntegerParameter(1, 10),        # igual
    'num_round': IntegerParameter(200, 800)             # igual
}

# Crear el tuner
tuner = HyperparameterTuner(
    estimator=xgb,
    objective_metric_name="validation:auc",
    hyperparameter_ranges=hyperparameter_ranges,
    max_jobs=50,
    max_parallel_jobs=8,
    objective_type="Maximize",
    base_tuning_job_name='hpo-plaft-pj-minorista'
)

# Entradas de datos
s3_input_train = TrainingInput(
    s3_data=f's3://{bucket_name}/{model_prefix}/data_dev_model/train_total.csv',
    content_type='csv'
)

s3_input_val = TrainingInput(
    s3_data=f's3://{bucket_name}/{model_prefix}/data_dev_model/validation_total.csv',
    content_type='csv'
)

# Ejecutar HPO
tuner.fit({'train': s3_input_train, 'validation': s3_input_val}, include_cls_metadata=False)


/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/estimator.py:588: SageMakerV2DeprecationWarning: Estimator is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `ModelTrainer` (`from sagemaker.train import ModelTrainer`).
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation(
Estimator is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `ModelTrainer` (`from sagemaker.train import ModelTrainer`).
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/tuner.py:689: SageMakerV2DeprecationWarning: HyperparameterTuner is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, u

...............................................................................................................................!
